# SEMANTIC GAP · BASELINE
## Myš · OBRAZ + TEXT

Samostatný experimentální běh bez MAPY VÝZNAMU. Používá jeden model Qwen3-VL-2B-Instruct. Výsledky a checkpointy se ukládají na Google Drive.

**Spuštění:** připoj Google Drive a spusť buňky shora dolů. Pokud chybí vstupní soubor, notebook nabídne nahrání. Po dokončení vytvoří ZIP a nabídne jeho stažení.

**Důležité:** tento notebook vytváří experimentální vektor z průměru skrytých stavů poslední vrstvy. Nejde o specializovaný embeddingový model. Neprovádí se fine-tuning ani změna vah.


## 1. Instalace knihoven


In [ ]:
!pip -q install -U "transformers>=4.57.0" accelerate safetensors pillow

## 2. Google Drive a nastavení běhu


In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, json, time, hashlib, shutil, zipfile, gc, textwrap, platform
from datetime import datetime, timezone

drive.mount("/content/drive")

# Nastavení tohoto samostatného běhu
ANIMAL_ID = "mys"
MODE = "combined"  # image | text | combined
MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
PROMPT_VERSION = "baseline-neutral-v1"

DRIVE_ROOT = Path("/content/drive/MyDrive/SEMANTIC_GAP_BASELINE")
INPUT_DIR = DRIVE_ROOT / "inputs"
RESULTS_ROOT = DRIVE_ROOT / "results"
EXPORT_ROOT = DRIVE_ROOT / "exports"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_PATH = INPUT_DIR / f"{ANIMAL_ID}.jpg"
TEXT_PATH = INPUT_DIR / f"{ANIMAL_ID}.txt"

print("Zvíře:", ANIMAL_ID)
print("Režim:", MODE)
print("Model:", MODEL_ID)
print("Vstupy:", INPUT_DIR)
print("Výsledky:", RESULTS_ROOT)
print("Tento běh nepoužívá MAPU VÝZNAMU ani topologické instrukce.")

## 3. Kontrola a načtení vstupů

Pro první kartu použij soubory `mys.jpg` a `mys.txt`. U textu se zachovává původní řádkování.


In [ ]:
from PIL import Image
import mimetypes

if MODE not in {"image", "text", "combined"}:
    raise ValueError("MODE musí být image, text nebo combined.")

required = []
if MODE in {"image", "combined"}:
    required.append(IMAGE_PATH)
if MODE in {"text", "combined"}:
    required.append(TEXT_PATH)

missing = [p for p in required if not p.exists()]
if missing:
    print("Nahraj chybějící soubory. Očekávané názvy:")
    for p in missing:
        print(" -", p.name)
    uploaded = files.upload()
    for filename, content in uploaded.items():
        destination = INPUT_DIR / Path(filename).name
        destination.write_bytes(content)
        print("Uloženo:", destination)

missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Nenašla jsem očekávané soubory: " + ", ".join(p.name for p in missing) +
        ". Nahraj je do složky SEMANTIC_GAP_BASELINE/inputs na Google Drive "
        "nebo je znovu nahraj přes dialog a použij přesný název."
    )

image = None
text_content = None
if MODE in {"image", "combined"}:
    image = Image.open(IMAGE_PATH).convert("RGB")
    print("Obraz:", image.size, IMAGE_PATH.name)
if MODE in {"text", "combined"}:
    # utf-8-sig odstraní případný BOM, ostatní řádkování zůstává zachováno.
    text_content = TEXT_PATH.read_bytes().decode("utf-8-sig")
    print("Text:", len(text_content), "znaků")
    print("Počet řádků:", len(text_content.splitlines()))
    print("Ukázka původního řádkování:")
    print(repr(text_content[:500]))

# Otisk vstupu umožňuje bezpečně navázat na správný checkpoint.
hash_obj = hashlib.sha256()
hash_obj.update(ANIMAL_ID.encode("utf-8"))
hash_obj.update(MODE.encode("utf-8"))
hash_obj.update(MODEL_ID.encode("utf-8"))
hash_obj.update(PROMPT_VERSION.encode("utf-8"))
if image is not None:
    hash_obj.update(IMAGE_PATH.read_bytes())
if text_content is not None:
    hash_obj.update(text_content.encode("utf-8"))
INPUT_FINGERPRINT = hash_obj.hexdigest()

RUN_DIR = RESULTS_ROOT / ANIMAL_ID / MODE / f"run_{INPUT_FINGERPRINT[:10]}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "input").mkdir(exist_ok=True)

print("Otisk vstupu:", INPUT_FINGERPRINT[:16])
print("Pracovní složka:", RUN_DIR)

## 4. Načtení jednoho multimodálního modelu

GPU se použije automaticky, pokud je Colab přidělí. Jinak se zkusí CPU. Model se při prvním běhu stáhne z Hugging Face.


In [ ]:
import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

HAS_GPU = torch.cuda.is_available()
DEVICE_KIND = "GPU" if HAS_GPU else "CPU"
DTYPE = torch.float16 if HAS_GPU else torch.float32

print("Výpočetní zařízení:", DEVICE_KIND)
if HAS_GPU:
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU paměť:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")
else:
    print("CPU režim může být výrazně pomalejší a může narazit na limit RAM Colabu.")

# Model se stáhne jen při prvním spuštění; další běhy využijí cache Colabu,
# pokud zůstane stejná runtime relace.
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()
print("Model načten.")

## 5. Neutrální zadání BASELINE

Model nedostává MAPU VÝZNAMU, autorské definice ani topologická pravidla. Stejné obecné zadání se používá ve všech třech režimech.


In [ ]:
# Neutrální baseline: žádné autorské definice ani MAPA VÝZNAMU.
SYSTEM_PROMPT = """You are a neutral research assistant. Analyze only the material supplied in this run.
Do not assume a private symbolic system, authorial definitions, or intended meanings that are not stated in the input.
Separate directly observable evidence from interpretation and speculation.
Preserve ambiguity and uncertainty. Do not invent missing context.
Do not provide hidden chain-of-thought. Give concise conclusions supported by evidence."""

USER_TASK = """Analyze the supplied material as a standalone item.
Identify directly observable content and formal or structural features.
Describe relationships among elements that are actually present in the supplied material.
Offer possible interpretations only as interpretations, not as facts.
List uncertainties and information that cannot be established from this input.
If visible writing appears inside an image, transcribe it when legible.
Return one valid JSON object only, using these keys:
{
  "visible_or_stated_content": [],
  "formal_and_structural_features": [],
  "relations_within_the_material": [],
  "possible_interpretations": [],
  "direct_evidence": [],
  "inferences": [],
  "uncertainties": []
}
Do not add authorial meanings that were not supplied."""

content = []
if MODE in {"image", "combined"}:
    content.append({"type": "image", "image": str(IMAGE_PATH)})
if MODE == "text":
    content.append({"type": "text", "text": USER_TASK + "\n\nMATERIAL (Czech original; preserve line breaks):\n" + text_content})
elif MODE == "image":
    content.append({"type": "text", "text": USER_TASK})
else:
    content.append({"type": "text", "text": USER_TASK + "\n\nSEPARATE CZECH STORY (not assumed to be a caption or description):\n" + text_content})

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": content},
]

print("Zpráva připravena. Režim:", MODE)
print("Obraz je součástí vstupu:", MODE in {"image", "combined"})
print("Samostatný text je součástí vstupu:", MODE in {"text", "combined"})

## 6. Výpočet embeddingu a checkpoint

Vektor se uloží hned po výpočtu. Pokud se další krok přeruší, opakované spuštění využije uložený embedding.


In [ ]:
import numpy as np
import torch.nn.functional as F

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def log_event(event, **details):
    record = {"time_utc": utc_now(), "event": event, **details}
    with (RUN_DIR / "worklog.jsonl").open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        # Drive checkpoint zapisujeme po každé události; fsync na FUSE mountu není nutný.

def save_json(path, obj):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(temp, path)

def load_metadata():
    p = RUN_DIR / "metadata.json"
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    return {}

metadata_path = RUN_DIR / "metadata.json"
embedding_path = RUN_DIR / "embedding.npy"
embedding_json_path = RUN_DIR / "embedding.json"

metadata = load_metadata()
same_run = metadata.get("input_fingerprint") == INPUT_FINGERPRINT
embedding_ready = same_run and embedding_path.exists() and embedding_json_path.exists()

if embedding_ready:
    vector = np.load(embedding_path)
    print("Embedding už existuje. Pokračuji z uloženého checkpointu:", vector.shape)
    log_event("embedding_checkpoint_reused", shape=list(vector.shape))
else:
    log_event("embedding_started", mode=MODE, model=MODEL_ID)
    start = time.time()

    model_inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    model_inputs = model_inputs.to(model.device)

    with torch.inference_mode():
        outputs = model(**model_inputs, output_hidden_states=True, use_cache=False, return_dict=True)

    hidden = outputs.hidden_states[-1]
    attention = model_inputs.get("attention_mask")
    if attention is None:
        attention = torch.ones(hidden.shape[:2], device=hidden.device, dtype=torch.long)

    # Mean pooling přes platné pozice poslední skryté vrstvy.
    # Ukládáme stejnou metodu pro všechny tři režimy.
    seq_len = min(hidden.shape[1], attention.shape[1])
    hidden = hidden[:, :seq_len, :].float()
    mask = attention[:, :seq_len].unsqueeze(-1).to(hidden.dtype)
    pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
    normalized = F.normalize(pooled, p=2, dim=-1)
    vector = normalized[0].detach().cpu().numpy().astype(np.float32)

    np.save(embedding_path, vector)
    save_json(embedding_json_path, {
        "animal": ANIMAL_ID,
        "mode": MODE,
        "model": MODEL_ID,
        "pooling": "mean_pool_last_hidden_state_attention_mask_l2_normalized",
        "vector_dimension": int(vector.shape[0]),
        "vector": vector.tolist(),
    })

    metadata = {
        "project": "Chceš myš? Slovník mezer",
        "phase": "BASELINE",
        "map_of_meaning_used": False,
        "animal": ANIMAL_ID,
        "mode": MODE,
        "model_id": MODEL_ID,
        "prompt_version": PROMPT_VERSION,
        "input_fingerprint": INPUT_FINGERPRINT,
        "pooling": "mean_pool_last_hidden_state_attention_mask_l2_normalized",
        "vector_dimension": int(vector.shape[0]),
        "sequence_length_used": int(seq_len),
        "device": DEVICE_KIND,
        "dtype": str(DTYPE),
        "transformers_version": __import__("transformers").__version__,
        "torch_version": torch.__version__,
        "embedding_seconds": round(time.time() - start, 3),
        "created_utc": utc_now(),
        "stage": "embedding_saved",
    }
    save_json(metadata_path, metadata)
    log_event("embedding_saved", shape=list(vector.shape), seconds=round(time.time()-start,3))
    print("Embedding uložen:", vector.shape)

for _name in ("outputs", "hidden", "pooled", "normalized", "model_inputs"):
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Checkpoint embeddingu:", embedding_path)

## 7. Strukturovaná analýza a pracovní záznam

Ukládá se pozorovatelný výstup modelu a časová posloupnost kroků. Nejde o záznam interního skrytého uvažování modelu.


In [ ]:
analysis_path = RUN_DIR / "analysis.json"
raw_analysis_path = RUN_DIR / "analysis_raw.txt"

if analysis_path.exists() and raw_analysis_path.exists():
    analysis = json.loads(analysis_path.read_text(encoding="utf-8"))
    print("Analýza už existuje. Přeskakuji dokončený krok.")
    log_event("analysis_checkpoint_reused")
else:
    log_event("analysis_started", mode=MODE)
    start = time.time()

    model_inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        generated = model.generate(
            **model_inputs,
            max_new_tokens=512,
            do_sample=False,
            use_cache=True,
        )

    prompt_length = model_inputs["input_ids"].shape[1]
    new_tokens = generated[:, prompt_length:]
    raw_output = processor.batch_decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    raw_analysis_path.write_text(raw_output, encoding="utf-8")

    cleaned = raw_output.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.replace("```json", "", 1).replace("```", "").strip()
    try:
        analysis = json.loads(cleaned)
        parse_success = True
    except Exception:
        analysis = {
            "parse_success": False,
            "raw_output": raw_output,
            "note": "Model nevrátil čistý JSON; původní výstup je zachován beze změny."
        }
        parse_success = False

    save_json(analysis_path, analysis)
    metadata = load_metadata()
    metadata.update({
        "analysis_seconds": round(time.time() - start, 3),
        "analysis_parse_success": parse_success,
        "analysis_saved_utc": utc_now(),
        "stage": "analysis_saved",
    })
    save_json(metadata_path, metadata)
    log_event("analysis_saved", parse_success=parse_success, seconds=round(time.time()-start,3))
    print("Analýza uložena. JSON parsování:", parse_success)
    print(raw_output[:4000])

for _name in ("generated", "model_inputs"):
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 8. ZIP archiv a stažení

Archiv zůstává na Google Drive a zároveň se nabídne ke stažení do zařízení.


In [ ]:
# Zkopírujeme původní vstupy do archivu, aby výsledek zůstal samostatně dohledatelný.
if MODE in {"image", "combined"}:
    shutil.copy2(IMAGE_PATH, RUN_DIR / "input" / IMAGE_PATH.name)
if MODE in {"text", "combined"}:
    shutil.copy2(TEXT_PATH, RUN_DIR / "input" / TEXT_PATH.name)

metadata = load_metadata()
metadata["stage"] = "complete"
metadata["completed_utc"] = utc_now()
save_json(metadata_path, metadata)
log_event("run_completed", animal=ANIMAL_ID, mode=MODE)

zip_name = f"{ANIMAL_ID}_{MODE}_{INPUT_FINGERPRINT[:10]}.zip"
zip_path = EXPORT_ROOT / zip_name
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for file_path in RUN_DIR.rglob("*"):
        if file_path.is_file():
            z.write(file_path, arcname=file_path.relative_to(RUN_DIR))

print("Hotovo.")
print("Výsledková složka:", RUN_DIR)
print("ZIP na Google Drive:", zip_path)
print("Velikost ZIP:", round(zip_path.stat().st_size / 1024, 1), "KB")

# Nabídne stažení ZIPu do zařízení. Kopie zůstává zároveň na Google Drive.
files.download(str(zip_path))